## Basic STM

In [ ]:
class SimpleSTM:
    def __init__(self):
        # Stores conversation history as (user, bot) pairs
        self.memory = []

    def add_interaction(self, user_input, llm_response):
        """Stores user question and LLM response in memory."""
        self.memory.append((user_input, llm_response))

    def get_context(self):
        """Formats stored memory as conversation context for LLM."""
        return "\n".join([f"User: {q}\nLLM: {a}" for q, a in self.memory])


# Example Usage
stm = SimpleSTM()
stm.add_interaction("What is AI?", "AI stands for Artificial Intelligence.")
stm.add_interaction("How does machine learning work?", "Machine learning is a subset of AI.")
context = stm.get_context()
print(context)

## Advanced STM

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "send_email",
        "description": "Send an email to a given recipient with a subject and message.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {
                    "type": "string",
                    "description": "The recipient email address."
                },
                "subject": {
                    "type": "string",
                    "description": "Email subject line."
                },
                "body": {
                    "type": "string",
                    "description": "Body of the email message."
                }
            },
            "required": [
                "to",
                "subject",
                "body"
            ],
            "additionalProperties": False
        },
        "strict": True
    }
}]

In [ ]:
from dataclasses import dataclass, field
from typing import List, Tuple, Callable, Optional, Dict
import re

# ---------- Pluggable hooks (replace with your production versions) ----------

def default_token_count(text: str) -> int:
    """
    Rough token estimator: ~1.3× words for plain text; a bit higher for code/JSON.
    Good enough to budget; swap for model tokenizer (e.g., tiktoken) in prod.
    """
    words = len(re.findall(r"\S+", text))
    # Heuristic bump for heavy punctuation / JSON-ish content
    punct_ratio = len(re.findall(r"[{}[\],:;]", text)) / max(1, len(text))
    coef = 1.3 + (0.2 if punct_ratio > 0.02 else 0.0)
    return int(coef * words)

def default_summarize(text: str, target_tokens: int) -> str:
    """
    Minimal abstractive stub: keeps the most informative sentences by length/density,
    then trims at a sentence boundary near target_tokens. Replace with your LLM call.
    """
    # Split into sentences and rank by 'informativeness' ≈ alphabetic characters count
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    ranked = sorted(sents, key=lambda s: len(re.findall(r"[A-Za-z0-9]", s)), reverse=True)
    out = []
    for s in ranked:
        out.append(s)
        if default_token_count(" ".join(out)) >= target_tokens:
            break
    summary = " ".join(out)
    # Soft trim on sentence boundary if we overshot
    while default_token_count(summary) > target_tokens and len(out) > 1:
        out.pop()
        summary = " ".join(out)
    return summary